# Ch 27. Flash Attention — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: PyTorch SDPA의 백엔드를 `math`, `mem_efficient`로 명시적 설정하고 비교하라.

### 풀이

같은 Q/K/V로 `math`와 `mem_efficient`를 각각 강제한다. backend가 현재 장치나 PyTorch build에서 지원되지 않으면 그 예외를 결과에 명시하고, 실행된 backend는 math 기준 최대 절대오차를 계산한다. 따라서 CPU에서 mem-efficient가 조용히 math로 fallback되는 일을 허용하지 않는다.

아래 코드는 다운로드 없이 실행되며 실제 backend 성공 여부와 수치 오차를 출력한다.


In [ ]:
import torch
from torch.nn.functional import scaled_dot_product_attention

torch.manual_seed(27)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
q = torch.randn(1, 2, 32, 16, device=device)

try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
    requested = {'math': SDPBackend.MATH, 'mem_efficient': SDPBackend.EFFICIENT_ATTENTION}
    def forced(backend):
        with sdpa_kernel(backend):
            return scaled_dot_product_attention(q, q, q)
except ImportError:  # PyTorch 2.0 compatibility
    from torch.backends.cuda import sdp_kernel
    requested = {'math': 'math', 'mem_efficient': 'mem_efficient'}
    def forced(backend):
        flags = {'enable_math': backend == 'math',
                 'enable_mem_efficient': backend == 'mem_efficient',
                 'enable_flash': False}
        with sdp_kernel(**flags):
            return scaled_dot_product_attention(q, q, q)

outputs, comparison = {}, {'device': device}
for name, backend in requested.items():
    try:
        outputs[name] = forced(backend)
        comparison[name] = {'status': 'executed', 'shape': tuple(outputs[name].shape)}
    except RuntimeError as error:
        comparison[name] = {'status': 'unsupported', 'reason': str(error).splitlines()[0]}

assert comparison['math']['status'] == 'executed'
if 'mem_efficient' in outputs:
    max_error = float((outputs['math'] - outputs['mem_efficient']).abs().max())
    comparison['mem_efficient']['max_abs_error_vs_math'] = max_error
    assert torch.allclose(outputs['math'], outputs['mem_efficient'], atol=2e-5, rtol=2e-5)
comparison


## 문제 2

**문제**: 시퀀스 길이 1024, 4096, 16384에서 표준 vs SDPA 시간과 메모리를 비교하라.

### 풀이

표준 attention은 $n\times n$ score를 저장해 메모리가 $O(n^2)$이다. 타일형 SDPA는 score materialization을 피하므로 작은 block workspace를 쓰지만, 실제 시간·메모리는 OOM을 결과로 포함해 장치별 측정해야 한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
lengths=[1024,4096,16384]; standard_scores=[n*n for n in lengths]; tiled_scores=[n*128 for n in lengths]
assert standard_scores[-1]/tiled_scores[-1]==128
list(zip(lengths,standard_scores,tiled_scores))


## 문제 3

**문제**: Online Softmax를 직접 구현하고 표준 소프트맥스와 결과가 같음을 보이라.

### 풀이

새 원소 $x$를 볼 때 $m' = \max(m,x)$, $l'=l e^{m-m'}+e^{x-m'}$로 갱신하면 overflow 없이 최종 분모를 얻는다. 두 구현의 확률 합과 원소별 오차를 검증한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
x=np.array([1000.,999.,998.,997.]); m=-np.inf; denom=0.
for v in x:
 new=max(m,v); denom=denom*np.exp(m-new)+np.exp(v-new); m=new
p=np.exp(x-m)/denom; ref=np.exp(x-x.max()); ref/=ref.sum()
assert np.allclose(p,ref); p


## 문제 4

**문제**: Flash Attention이 역전파 메모리를 어떻게 절약하는지 설명하라.

### 풀이

Flash Attention은 순전파의 전체 score/확률 행렬을 저장하지 않고 block별로 재계산한다. 계산량 일부를 더 써서 activation 저장을 $O(n^2)$에서 대략 $O(nd)$로 낮추는 compute-memory tradeoff다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
def memory_elements(n,d,block): return {'standard_scores':n*n,'flash_workspace':block*block+n*d}
r=memory_elements(4096,64,128); assert r['flash_workspace']<r['standard_scores']; r


## 문제 5

**문제**: Ring Attention이 다중 GPU에서 어떻게 동작하는지 설명하라.

### 풀이

각 장치는 로컬 Q를 유지하고 K/V block을 ring으로 $N-1$번 전달한다. 매 단계 online-softmax 통계를 병합하면 모든 K/V를 본 정확한 attention을 얻고, 장치당 저장은 전체의 약 $1/N$이다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
devices=4; blocks=list(range(devices)); seen=[]
for step in range(devices): seen.append([blocks[(rank-step)%devices] for rank in range(devices)])
assert all(set(seen[s])==set(range(devices)) for s in range(devices)); seen
